# FedX-Palm v2 -- YOLOv11 Federated Learning + Differential Privacy + XAI

Notebook ini menjalankan seluruh pipeline FedX-Palm end-to-end: download dataset baru
dari Roboflow, split leakage-free, partisi Non-IID (Dirichlet) ke K klien, lalu empat
blok eksperimen **B1 (centralized baseline)**, **B2 (federated, tanpa DP)**,
**E1 (DP-SGD federated penuh)**, **E2 (DP-SGD federated parsial, backbone dibekukan)**,
diakhiri evaluasi XAI (Grad-CAM++: Average Drop & Focus Retention Rate) dan ekspor
tabel hasil untuk Bab 4-5 tesis.

**Jalankan di GPU** (Colab GPU runtime, atau server RTX kamu sendiri via Jupyter).
Semua konfigurasi (K values, sigma sweep, epoch, dst) ada di `configs/*.yaml` --
ubah di sana, bukan di notebook ini, supaya konsisten dengan apa yang didokumentasikan
di Bab 3 tesis.

**Estimasi waktu**: sweep penuh B2 (5 nilai K x 40 ronde) + E1/E2 (5 K x 5 sigma x 40
ronde x 2 varian) itu berat -- ratusan run training. Jalankan dulu dengan
`--k` / `--sigma` di-subset (lihat sel opsional di tiap blok) untuk sanity check
sebelum commit ke sweep penuh semalaman.


## 0. Setup lingkungan

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: tidak ada GPU terdeteksi -- training akan sangat lambat.")


Jika repo belum ter-clone di environment ini (mis. Colab fresh runtime), jalankan sel
berikut. Kalau sudah berada di dalam clone repo (mis. Jupyter di server RTX), lewati saja.

In [ ]:
# %cd /content
# !git clone -b final-tesis-2026 https://github.com/rachmadiantyy/fedx-palm.git
# %cd fedx-palm


In [ ]:
!pip install -q -r requirements.txt


## 1. Download dataset dari Roboflow

Membaca konfigurasi dari `configs/dataset.yaml` (workspace, project, version sudah diisi).

In [ ]:
!python scripts/01_download_dataset.py


## 2. Split leakage-free (berbasis bunch_id)

Menggabungkan train/valid/test bawaan Roboflow lalu memecah ulang berdasarkan identitas
tandan (bukan per-frame acak), supaya tidak ada foto dari tandan yang sama menyebar ke
train dan test sekaligus.

In [ ]:
!python scripts/02_prepare_splits.py


## 3. Partisi Non-IID (Dirichlet) ke K klien

Untuk setiap K di `configs/fl_config.yaml` (default: 2, 4, 8, 12, 16), partisi data train
lalu materialisasi folder per-klien (symlink + data.yaml).

In [ ]:
!python scripts/03_partition_clients.py


## 4. Siapkan model dasar (BatchNorm -> GroupNorm)

Semua 81 layer BatchNorm dikonversi ke GroupNorm di sini, sekali, lalu dipakai sebagai
titik awal yang identik untuk B1/B2/E1/E2 -- supaya perbandingan antar blok eksperimen adil.

In [ ]:
!python scripts/04_prepare_base_model.py


## 5. B1 -- Baseline Centralized

Fine-tune di seluruh data train (tanpa federasi, tanpa DP). Ini jadi acuan atas (upper
bound) untuk dibandingkan dengan hasil federasi/privat di langkah berikutnya.

In [ ]:
!python scripts/05_train_b1_centralized.py --device 0


## 6. B2 -- Federated (FedAvg, tanpa DP)

Sweep semua K di `configs/fl_config.yaml`. Untuk uji cepat satu K dulu, pakai `--k 4`.

In [ ]:
# Uji cepat (opsional, satu K + ronde sedikit):
# !python scripts/06_train_b2_federated.py --device 0 --k 4 --rounds 5

!python scripts/06_train_b2_federated.py --device 0


## 7. E1 -- DP-SGD Federated Penuh

Sweep grid K x sigma penuh (`configs/dp_config.yaml`). epsilon dihitung otomatis oleh
PRV accountant Opacus, bukan ditetapkan manual.

In [ ]:
# Uji cepat (opsional, satu sel grid):
# !python scripts/07_train_e1_dp_full.py --device 0 --k 4 --sigma 1.0 --rounds 5

!python scripts/07_train_e1_dp_full.py --device 0


## 8. E2 -- DP-SGD Federated Parsial (backbone dibekukan)

In [ ]:
# Uji cepat (opsional):
# !python scripts/08_train_e2_dp_partial.py --device 0 --k 4 --sigma 1.0 --rounds 5

!python scripts/08_train_e2_dp_partial.py --device 0


## 9. Evaluasi XAI (Grad-CAM++: Average Drop & Focus Retention Rate)

Pilih checkpoint yang mau dijadikan "model operasional" (mis. B2 di K yang paling
mendekati baseline terpusat -- lihat `results/table_4_2_b2_federated.csv` setelah
sel berikutnya untuk menentukan K terbaik).

In [ ]:
import json, glob

# Contoh: pilih checkpoint B2 K=4 (sesuaikan dengan K terbaikmu setelah lihat hasil)
with open("results/b2_k4.json") as f:
    b2_k4 = json.load(f)
weights_path = b2_k4["weights"]
print("Evaluasi XAI pada:", weights_path)


In [ ]:
import subprocess

subprocess.run(["python", "scripts/09_evaluate_xai.py", "--weights", weights_path,
                 "--tag", "b2_k4", "--save-overlays"], check=True)


## 10. Ekspor semua hasil ke tabel CSV (untuk Bab 4-5)

In [ ]:
!python scripts/10_export_results.py


## 11. Kurva privasi-utilitas (mirip Gambar 4.1/4.2 tesis)

Plot cepat mAP@0.5 vs epsilon per K, dari hasil E1.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results/table_4_3_e1_dp_full.csv")
fig, ax = plt.subplots(figsize=(6, 4))
for k, g in df.groupby("k"):
    g = g.sort_values("epsilon")
    ax.plot(g["epsilon"], g["map50"], marker="o", label=f"K={k}")
ax.set_xlabel("epsilon (privacy budget)")
ax.set_ylabel("mAP@0.5")
ax.set_title("E1: privacy-utility curve")
ax.legend()
plt.show()


## 12. Kumpulkan semuanya untuk dikirim balik

Zip folder `results/` (JSON + CSV + overlay Grad-CAM++) dan kirimkan isinya (atau file
zip-nya) supaya Bab 4 & 5 tesis bisa ditulis berdasarkan angka asli dari run ini.

In [ ]:
!zip -r fedxpalm_results.zip results/
print("Selesai -- unduh fedxpalm_results.zip. Simpan arsip hasil eksperimen untuk keperluan evaluasi dan dokumentasi.")
